In [1]:
# Typical steps
# 1. Application parses policy documents and extracts relevant information to create a knowledge base.

# This data will be stored in vector database and will be used as context for question answering.
content_corpus = [
        "Employees are compensated on a bi-weekly basis through direct deposit.",       
        "Employees must submit a leave request for approval.", 
        "Company internet must be used for work-related tasks only.",
        "Company internet is a broadband internet.",
        "Employees can take an hour break.",
        "Interact with each employee with Respect"
]


In [2]:
# Goal: Retrieve relevant documents based on user query (a.k.a. Context Retrieval)

%pip install -q sentence-transformers


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = model.encode(content_corpus)

/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9150.69it/s]


In [3]:
doc_vectors

array([[-0.01192882,  0.01236366,  0.03582982, ...,  0.01726838,
         0.07261266, -0.0317501 ],
       [ 0.03315504,  0.04853383,  0.04736274, ...,  0.10182008,
         0.09159282,  0.00358369],
       [-0.07135904, -0.03066471,  0.03183771, ..., -0.04109798,
         0.06524782, -0.00688535],
       [-0.00383736, -0.02336751,  0.02958677, ..., -0.04415291,
         0.12559088, -0.03139851],
       [-0.01790445,  0.0149585 ,  0.08163826, ..., -0.03217234,
        -0.00513649,  0.05279536],
       [-0.00240888,  0.03361145, -0.06162645, ...,  0.04830879,
         0.03707644, -0.01683046]], shape=(6, 384), dtype=float32)

In [7]:
# Step 3: User Query and Semantic Matching

query = "What’s the wifi policy?"
query_vec = model.encode([query])[0]
query_vec

array([-9.56225954e-03,  2.66804378e-02,  2.46216934e-02, -3.85921844e-03,
        2.13141367e-02,  4.90859412e-02,  4.35561910e-02, -5.60238473e-02,
       -3.68580408e-02,  2.62932032e-02,  6.04454428e-02,  1.25224024e-01,
        1.05912834e-02, -3.88664119e-02,  1.97370797e-02,  2.03328077e-02,
        2.59528216e-02, -1.36639699e-01, -8.37292746e-02, -8.39904621e-02,
        8.78420472e-02, -3.93093005e-03,  7.46909082e-02, -1.17777195e-02,
        1.10653587e-01, -9.13339760e-03,  1.04997061e-01,  4.71126363e-02,
       -6.25546873e-02,  7.02960491e-02, -1.55354966e-04, -4.56475019e-02,
        5.82433026e-03, -5.62751740e-02, -6.38677329e-02, -9.21166688e-02,
       -8.46939161e-02,  1.67320799e-02, -3.05885002e-02,  4.60009463e-03,
       -4.37936261e-02, -4.89386469e-02, -5.51083125e-02,  1.25143856e-01,
        3.76796313e-02,  4.22913432e-02,  4.14212653e-03,  2.78568901e-02,
        1.68612432e-02, -5.36200777e-02,  9.44720730e-02,  1.33927744e-02,
        7.85079692e-03,  

In [26]:
similarities = model.similarity(query_vec, doc_vectors)
print(type(similarities))

# Ensure it's a 1D numpy array
import numpy as np
similarities = np.asarray(similarities).squeeze()
similarities

<class 'torch.Tensor'>


array([0.0968748 , 0.12412912, 0.3093428 , 0.36806744, 0.13711047,
       0.01130453], dtype=float32)

In [27]:

# Now get top 3
top_3_indices = np.argsort(similarities)[::-1][:3]
print(top_3_indices)
top_scores = similarities[top_3_indices]
top_scores

[3 2 4]


array([0.36806744, 0.3093428 , 0.13711047], dtype=float32)

In [28]:
top_scores

array([0.36806744, 0.3093428 , 0.13711047], dtype=float32)

In [29]:
top_docs = [documents[i] for i in top_3_indices]


print (top_docs)
context = ", ".join(top_docs)
context

['Company internet is a broadband internet.', 'Company internet must be used for work-related tasks only.', 'Employees can take an hour break.']


'Company internet is a broadband internet., Company internet must be used for work-related tasks only., Employees can take an hour break.'

In [30]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

my_client = OpenAI(api_key=my_api_key)
# my_client

def ask_question_open_ai(prompt):

    # print(f"User asked: {prompt}")
    # my_client.chat.completions.create

    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        # messages=[
        #     {"role": "system", "content": "You are a helpful assistant. Answer as concisely as possible."},
        #     {"role": "user", "content": prompt}
        # ]
        messages=[
            {"role": "system", "content": '''
             You are an assistant who answers only based on the given context.
             '''},
            {"role": "user", "content": f"Context: {context}\n\n User Question: {query}"} 
        ]

    )
    return llm_response.choices[0].message.content  


In [11]:
print (query)
response = ask_question_open_ai(query)

What’s the internet usage policy?


In [12]:
print(f"User query: {query}")
print(f"Context: {context}")

print(f"\n\nOpen AI Response: {response}")

User query: What’s the internet usage policy?
Context: Company internet is a broadband internet., Company internet must be used for work-related tasks only., Employees can take an hour break.


Open AI Response: - Internet type: broadband provided by the company.
- Allowed use: must be used for work-related tasks only.
- Breaks: employees may take up to one hour for a break.


In [13]:
#Reference Data
expected_answer = "The company's internet must be used for work-related tasks only."    
actual_response = response

# Call ChatCompltions API to compare expected vs actual
#LLM AS A JUDGE
comparison_prompt = f"""
You are an expert evaluator. Compare the actual response to the expected answer and determine if they match in meaning.
Expected Answer: {expected_answer}
Actual Response: {actual_response}
Do they match in meaning? Answer with 'Yes' or 'No' and provide a brief explanation.
"""
comparison_response = my_client.chat.completions.create(
    model="gpt-5-nano",     
    messages=[
        {"role": "system", "content": "You are an expert evaluator."},
        {"role": "user", "content": comparison_prompt}
    ]
)   
print(f"\nExpected Answer: {comparison_response.choices[0].message.content}")


Expected Answer: Yes — both express that internet use is limited to work-related tasks. The actual response includes extra details (internet type and breaks) but does not contradict the work-only rule.


In [ ]:
print (f"\nExpected Answer: {expected_answer}")
print (f"\nActual Response: {actual_response}")